# Lecture 02: ejercicio — aprender un formato de respuesta con LoRA y comparar antes y después del entrenamiento

Usaremos una FAQ ficticia de un curso para entrenar brevemente con LoRA un modelo pequeño ya ajustado para seguir instrucciones y comparar **qué cambia y qué no cambia** antes y después. La evaluación distingue entre la **tasa de cumplimiento del formato** (si se respeta el JSON con `answer` y `needs_review`) y la **tasa de acierto del contenido**.

**Qué comprobaremos en este ejercicio**

1. Un modelo pequeño puede incumplir el formato aunque se lo indiquemos en el prompt (sección 3, antes del entrenamiento).
2. En el experimento original en japonés, tras unos minutos de entrenamiento con LoRA, el modelo llegó a respetar el formato casi siempre. Comprobaremos el resultado con los datos traducidos (sección 5, después del entrenamiento).
3. Conseguir el formato no garantiza que el modelo sepa cuándo pedir confirmación y cuándo responder con certeza ante preguntas ausentes de la FAQ (secciones 5 y 6).
4. Avanzado: cargar el mismo modelo con cuantización en 4 bits cambia las salidas incluso sin entrenarlo (sección 8).

**Procedimiento**

| Sección | Contenido | Tiempo aproximado (T4) |
| --- | --- | --- |
| 0–3 | Entorno, configuración, generación de datos, carga del modelo y evaluación previa | Unos 5 minutos |
| 4–7 | Entrenamiento, evaluación posterior, preguntas de conocimiento, tabla comparativa y entrega | Unos 10 minutos (incluidos 4–5 de entrenamiento en el experimento original en japonés) |
| 8 (opcional) | Carga con cuantización en 4 bits y comparación | Unos 2 minutos |

Ejecutar las secciones 0–3 antes de la clase para comprobar el entorno ayuda a evitar problemas durante la sesión (opcional). En Colab, las variables se pierden al desconectarse la sesión; en ese caso, el día de clase vuelve a ejecutar desde la sección 0.

**Herramientas**: Google Colab (GPU T4, cuota gratuita). Modelo: `Qwen/Qwen3.5-0.8B` (Apache-2.0). El cuaderno genera datos ficticios que no incluyen plazos ni URL reales.

**Cómo leer el cuaderno**: las explicaciones de cada sección indican «Qué hacer», «Qué observar» y «Si algo falla». Ejecuta las celdas de arriba abajo. Si omites alguna, las siguientes pueden fallar por no encontrar una variable.

**Nota sobre las observaciones**: los tiempos, las pérdidas y los comportamientos de salida descritos proceden del ejercicio original en japonés. Pueden variar con la FAQ y las preguntas traducidas.

Versión: 1.1 (2026-10-18)


## 0. Comprobar el entorno e instalar las bibliotecas

**Qué hacer**: comprueba que se te haya asignado una GPU e instala las bibliotecas con versiones fijadas cuyo funcionamiento se ha verificado.

**Qué observar**

- Que la primera celda muestre `Tesla T4`.
- Que al final de la segunda celda aparezca la lista de versiones, por ejemplo `transformers: 5.18.0`. La instalación tarda aproximadamente 1 minuto.

**Si algo falla**

- Si aparece «No se ha encontrado una GPU»: selecciona **Runtime → Change runtime type → T4 GPU** y vuelve a ejecutar la celda. Si no se te asigna una por saturación, inténtalo más tarde.
- El mensaje rojo de pip `ERROR: pip's dependency resolver ...` (sobre `cudf` o `pyarrow`) advierte de incompatibilidades con otros paquetes preinstalados en Colab. No afecta a este ejercicio, por lo que puedes continuar.


In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # Los entornos de ejecución CPU no tienen nvidia-smi
    gpu_info = ""
print(gpu_info or "No se ha encontrado una GPU. Selecciona Runtime → Change runtime type → T4 GPU.")

In [ ]:
# Versiones verificadas en una T4 de Colab (Python 3.13, torch 2.11.0+cu130). Se usa torch tal como viene en Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requiere transformers 5
    "peft": "0.21.1",           # Implementación de LoRA
    "trl": "1.14.2",            # Bucle de entrenamiento (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # torchao (0.10) de Colab es incompatible con peft 0.21 y causa ImportError al entrenar. Se elimina porque no se usa
%pip install -q {spec}

# Las advertencias de dependencias de cudf-cu13 y pyarrow aparecen como ERROR, pero se pueden ignorar porque no se usa cudf
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Configuración

**Qué hacer**: reúne el nombre del modelo, el número de épocas de entrenamiento (pasadas por los datos) y otros ajustes en un único diccionario `CONFIG`. Se guardará íntegramente en el archivo de entrega de la sección 7 para que después puedas saber «en qué condiciones se midió».

- Fijamos `revision` para usar siempre los mismos pesos. Así se mantienen las condiciones al comparar antes y después del entrenamiento o con otras personas.
- `num_train_epochs: 20` indica 20 pasadas por los datos de entrenamiento (14 ejemplos). Cada pasada tiene 2 pasos, para un total de 40 pasos; en el experimento original en japonés, el entrenamiento tardó 4–5 minutos en una T4. El tiempo con los datos traducidos puede variar.
- `gen: {"do_sample": False}` indica generación voraz. Mantiene las mismas condiciones de generación antes y después y elimina el efecto de la aleatoriedad.

Si estableces `USE_DRIVE` en `True`, se monta Google Drive y se guardan allí también el adaptador entrenado y los archivos de resultados. Úsalo si quieres conservar los resultados aunque se desconecte la sesión. Si vas a terminar todo en una sesión, puedes dejarlo en `False`.

**Qué observar**: se muestra el JSON de configuración.


In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Modelo pequeño ajustado para seguir instrucciones (Apache-2.0). Cabe en una T4 sin cuantización
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Fijada para usar siempre los mismos pesos
    "seed": 42,
    "num_train_epochs": 20,  # Número de pasadas por los datos de entrenamiento (14 ejemplos). 2 pasos por pasada: 40 pasos en total (4–5 minutos en una T4 en el experimento original en japonés)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # Dimensión de las matrices de LoRA (cuanto menor, menos parámetros se entrenan)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # Evaluación fijada a generación voraz (mismas condiciones antes y después)
    "work_dir": "/content/qlora_lab",                          # Destino de los resultados (visible en la lista de archivos de la izquierda)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Destino cuando USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Con True, monta Google Drive y guarda el adaptador y las salidas
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. Generar la FAQ ficticia y los datos supervisados

**Qué hacer**: a partir de una FAQ ficticia de 5 entradas, crea los datos de entrenamiento (train), validación (valid) y evaluación (test).

El modelo recibe «FAQ + pregunta» y debe devolver una sola línea de JSON: `{"answer": <cadena>, "needs_review": <booleano>}`.

- Para preguntas con evidencia en la FAQ, responde con el texto de la FAQ y establece `needs_review: false`.
- Para preguntas ausentes de la FAQ, responde «No se indica en esta FAQ. Se necesita confirmación de la organización.» y establece `needs_review: true`.

**División de los datos**

- Las reformulaciones de preguntas sobre una misma entrada de la FAQ se reparten entre train / valid / test **por plantilla**. Así, las frases vistas en el entrenamiento no aparecen sin cambios en test.
- test contiene 21 preguntas (15 con evidencia y 6 sin evidencia). Para alcanzar esa cantidad, preparamos preguntas adicionales que solo se usan en test.
- Las respuestas correctas (el valor de `needs_review` y las expresiones que debe contener la respuesta) se guardan aparte en `gold` y no se pasan al modelo.
- train contiene 14 ejemplos (10 con evidencia y 4 sin evidencia); valid contiene 7. Como solo hay 5 entradas de FAQ, los datos de entrenamiento son pocos y se entrena varias veces con los mismos ejemplos. El número de repeticiones se decide con el número de épocas (`num_train_epochs`) de la sección 4, sin duplicar los datos. La regla general es guardar los ejemplos sin duplicarlos en los archivos de datos y definir la cantidad de entrenamiento en su configuración.

En el nombre de ejemplo `student_id_assignment_number.pdf`, sustituye `student_id` por tu identificador de estudiante y `assignment_number` por el número de la tarea; conserva la extensión `.pdf`.

**Qué observar**: se muestran `train=14 valid=7 test=21 gold=21` y un ejemplo de entrenamiento (pregunta y salida deseada). Si una pregunta se repite en train / valid / test, se produce un error en esta celda.


In [ ]:
FAQ = {
    "deadline": "El plazo de entrega de la tarea es el 30 de septiembre a las 23:59 (hora de Japón).",
    "destination": "La entrega se realiza en la sección «Tareas» del portal de aprendizaje.",
    "format": "Debes entregar 1 archivo PDF con el nombre student_id_assignment_number.pdf.",
    "office_hours": "Puedes hacer preguntas en el horario de consulta de los martes de 17:00 a 18:00.",
    "late": "Se aceptan entregas hasta 48 horas después del plazo, con una penalización en la nota.",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# Con evidencia: (clave de FAQ, plantillas de preguntas, expresiones requeridas en el answer correcto)
GROUNDED = {
    "deadline": (["¿Cuál es el plazo de entrega de la tarea?", "¿Hasta cuándo puedo entregar?", "Indica la fecha límite.", "¿Cuándo vence la entrega?"], ["30 de septiembre", "23:59"]),
    "destination": (["¿Dónde se entrega?", "¿Dónde lo subo?", "¿Desde dónde entrego la tarea?", "¿Cuál es el lugar de entrega?"], ["portal de aprendizaje", "Tareas"]),
    "format": (["Indica el formato de entrega.", "¿Hay alguna regla para el nombre del archivo?", "¿En qué formato de archivo lo entrego?", "¿Puedo entregar en un formato distinto de PDF?"], ["PDF"]),
    "office_hours": (["¿Cuándo puedo hacer preguntas?", "¿Cuándo es el horario de consulta?", "¿Hay un horario para hacer preguntas al docente?", "¿En qué horario puedo consultar dudas?"], ["martes", "17:00"]),
    "late": (["¿Qué pasa si entrego tarde?", "¿Se acepta una entrega fuera de plazo?", "¿Cuál es la norma para las entregas tardías?", "Explica cómo se tratan las entregas después de la fecha límite."], ["48 horas", "penalización"]),
}
# Sin evidencia: no aparece en la FAQ; la solución es needs_review=true
UNGROUNDED = [
    "¿Puedo volver a entregar?", "Explica cómo se trata una ausencia a clase.", "¿Puedo ver las notas de otros participantes?",
    "¿Cuándo se publican las notas?", "¿Se puede ampliar el plazo de entrega?", "¿Se puede entregar en grupo?",
    "¿Cómo se distribuyen los puntos de la tarea?", "¿Se publicarán las grabaciones de las clases?",
]

# Preguntas adicionales solo para test (no se incluyen en train / valid)
TEST_EXTRA_GROUNDED = {
    "deadline": ["Indica la fecha y la hora del plazo de entrega.", "¿Qué día, mes y hora vence la tarea?"],
    "destination": ["¿Dónde se realiza la entrega de la tarea?", "Indica dónde hay que presentar el trabajo."],
    "format": ["¿Cuál es el formato del archivo que se entrega?", "¿En qué formato y cuántos archivos debo entregar?"],
    "office_hours": ["Indica el día y la hora del horario de consulta.", "¿Qué día y a partir de qué hora se atienden las preguntas?"],
    "late": ["¿Qué ocurre si no entrego antes de la fecha límite?", "¿Se acepta una tarea entregada después del plazo?"],
}
TEST_EXTRA_UNGROUNDED = [
    "¿Hay un límite de caracteres para la tarea?", "¿Recibiré comentarios sobre la tarea entregada?",
    "¿Se exige algún formato para las referencias bibliográficas?", "¿Hay que reservar antes para acudir al horario de consulta?",
]
UNGROUNDED_ANSWER = "No se indica en esta FAQ. Se necesita confirmación de la organización."

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # Primeras 2 plantillas para train, tercera para valid y cuarta para test (sin compartir reformulaciones)
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # De cada 4 preguntas sin evidencia, asigna 2 a train, 1 a valid y 1 a test
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["confirmación"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["confirmación"]}

# train contiene 14 ejemplos (10 con evidencia y 4 sin ella). Las repeticiones se deciden con num_train_epochs de la sección 4, sin duplicar datos
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "Hay preguntas repetidas entre train / valid / test. Revisa las plantillas"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("Ejemplo de entrenamiento | Pregunta:", train[0]["question"], "| Salida deseada:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Cargar el modelo y evaluar antes del entrenamiento

**Qué hacer**: carga el modelo base sin cuantización (32 bits), haz que responda a las 21 preguntas de test y puntúa las respuestas.

El prompt (`SYSTEM`) indica «devuelve siempre `answer` y `needs_review` en una sola línea de JSON». Es decir, **el formato ya está indicado**. Antes de entrenar, comprobamos si el modelo puede seguir esa instrucción.

**Criterios de puntuación** (en la sección 5 se usan las mismas funciones)

- **Formato**: se puede leer como JSON, tiene exactamente las dos claves `answer` y `needs_review`, y sus tipos son cadena y booleano.
- **Contenido**: `needs_review` coincide con la solución y `answer` contiene las expresiones requeridas, por ejemplo «30 de septiembre» y «23:59».

Para aprobar el contenido, `needs_review` debe coincidir. Por eso, si falta una clave, el contenido se marca como «Incorrecto» aunque el texto sea correcto. Comprueba el texto por tu cuenta en la tabla.

En la columna del motivo del fallo de formato, `json_error` significa que el texto no se puede leer como JSON; `not_object`, que el JSON no es un objeto; `extra_keys`, que contiene claves adicionales; `missing_key`, que falta una clave obligatoria; y `wrong_type`, que un valor tiene un tipo incorrecto.

`Qwen/Qwen3.5-0.8B` también puede procesar imágenes, pero al cargarlo con `AutoModelForCausalLM` solo se usa la parte de texto.

**Qué observar**

- Primera celda: aparece algo como `Memoria de los pesos (GB): 3.01`. La descarga del modelo (unos 1.75GB) tarda aproximadamente 1 minuto.
- Segunda celda: un resumen de una línea (`format_rate`, `content_rate`, etc.) y la tabla de las salidas de las 21 preguntas. La evaluación tarda alrededor de 1 minuto. En el experimento original en japonés, faltaba `needs_review` en muchas salidas antes del entrenamiento; comprueba qué ocurre con los datos traducidos.

**Advertencias que no impiden continuar**

- `You are sending unauthenticated requests to the HF Hub`: indica que se descarga el modelo sin iniciar sesión en Hugging Face. Los modelos públicos se pueden descargar sin iniciar sesión. La necesidad de iniciar sesión depende de los siguientes casos, no del tamaño del modelo.
  - Usar un modelo que exige aceptar sus condiciones (gated model, como Llama o Gemma) o un repositorio privado.
  - Subir tus propios modelos o datos al Hub.
  - Enviar muchas solicitudes en poco tiempo y alcanzar el límite. Sin iniciar sesión, el límite es de 3000 descargas de archivos cada 5 minutos por dirección IP (cifra publicada en septiembre de 2025; [límites de uso del Hub](https://huggingface.co/docs/hub/rate-limits)); aumenta al iniciar sesión.

  El modelo de este cuaderno es público y no requiere aceptar condiciones. La carga envía como máximo unas decenas de solicitudes, por lo que no se alcanza el límite. Si aun así aparece `429 Too Many Requests`, la biblioteca lee el tiempo de espera y reintenta automáticamente. Si necesitas iniciar sesión, registra un token de Hugging Face con el nombre `HF_TOKEN` en el icono de la llave (Secrets) a la izquierda de Colab y permite que este cuaderno acceda a él; se usará automáticamente.
- `... is falling back to its reference PyTorch implementation ...`: indica que no está instalada la implementación rápida de algunas capas (atención lineal), por lo que se usa otra más lenta pero correcta. Aparecen unas 4 líneas.

**Qué debes leer ahora**: observa la tabla y comprueba «qué claves faltan» y «si el texto coincide con la FAQ». En la sección 5, esta tabla se comparará con las salidas posteriores al entrenamiento. Las salidas originales también se guardan en `/content/qlora_lab/eval_before.json` (puedes abrirlo desde la lista de archivos de la izquierda).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# Sin cuantización. Mantiene los pesos en 32 bits (dtype es explícito para evitar usar la precisión de distribución)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("Modelo cargado:", CONFIG["base_model"], "| Dispositivo:", DEVICE, "| Memoria de los pesos (GB):", BASE_MEMORY_GB)

# El prompt ya indica el formato. Antes de entrenar se comprueba si el modelo lo respeta
SYSTEM = "Respondes a las preguntas de la FAQ del curso. Usa únicamente la FAQ proporcionada como evidencia y responde siempre en una sola línea de JSON. Formato: {\"answer\": cadena, \"needs_review\": booleano}. Si algo no se indica en la FAQ, escribe «Se necesita confirmación» en answer y establece needs_review en true."

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\nPregunta: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: deja vacío el bloque de razonamiento y genera solo la respuesta (explícito para mantener las condiciones al cambiar de modelo)
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Mostrar las salidas originales sin truncarlas

def check_format(text: str):
    '''Comprueba si es JSON válido con las claves y los tipos requeridos. Devuelve también el motivo del fallo'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Contenido correcto: needs_review coincide con la solución y answer contiene las expresiones requeridas'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "Correcto" if ok else "Incorrecto"

def evaluate(model, label: str):
    '''Genera y puntúa las 21 preguntas de test, y devuelve un resumen y los resultados por pregunta. Guarda las salidas en eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # Si puede leerse un fragmento similar a JSON aunque falle el formato, puntúa el contenido por separado
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# Salidas por pregunta. Compara con «needs_review correcto» y observa qué claves faltan y si el texto coincide con la FAQ
display(pd.DataFrame([{"Pregunta": r["question"], "needs_review correcto": r["gold_needs_review"], "Salida antes del entrenamiento": r["raw"],
                       "Formato": mark(r["format_ok"]), "Motivo del fallo de formato": r["format_reason"], "Contenido": mark(r["content_ok"])} for r in before_rows]))

## 4. Entrenamiento con LoRA (4–5 minutos en una T4 en el experimento original en japonés)

### Diferencia entre LoRA y QLoRA

- **LoRA**: mantiene fijos los pesos del modelo fundacional y entrena únicamente pequeñas matrices (adaptadores) añadidas a algunas capas.
- **QLoRA**: mantiene los pesos del modelo fundacional cuantizados en 4 bits y entrena solo los adaptadores de la misma forma. Los adaptadores no se cuantizan.

QLoRA reduce la memoria necesaria para mantener los pesos del modelo fundacional. Ocupan aproximadamente una cuarta parte de lo que ocupan en 16 bits, lo que permite entrenar modelos que de otro modo no cabrían en la GPU (7000 millones de parámetros ocupan unos 14GB en 16 bits y unos 3.5GB en 4 bits).

| Aspecto | LoRA (sin cuantización) | QLoRA (4 bits) |
| --- | --- | --- |
| Memoria de los pesos | Mayor | Menor (aproximadamente 1/4 de 16 bits) |
| Salida del modelo fundacional | Se conserva | Puede cambiar por los errores de cuantización; el efecto depende del modelo |
| Velocidad de entrenamiento | Referencia | No aumenta. Suele disminuir porque los pesos de 4 bits se reconstruyen en cada cálculo |
| Bibliotecas necesarias | peft | peft y bitsandbytes (requiere GPU CUDA) |

Consideraciones al implementar QLoRA:

- Pasa `BitsAndBytesConfig(load_in_4bit=True, ...)` al cargar y ejecuta `prepare_model_for_kbit_training(model)` antes de añadir LoRA.
- La precisión de almacenamiento (4 bits) y la de cálculo (`bnb_4bit_compute_dtype`, por ejemplo 16 bits) son ajustes distintos.
- Compara antes y después del entrenamiento con el modelo fundacional en las mismas condiciones de cuantización. Si evalúas antes sin cuantización y después en 4 bits, se mezclan los efectos de la cuantización y del entrenamiento.
- El adaptador se crea para el modelo fundacional usado al entrenar. Al distribuirlo o reutilizarlo, registra también el ID, la revisión y la configuración de cuantización del modelo base.
- Algunos modelos no recomiendan el entrenamiento en 4 bits. Una [guía de una biblioteca de entrenamiento](https://unsloth.ai/docs/models/qwen3.5/fine-tune) desaconseja entrenar Qwen3.5 en 4 bits porque sus errores de cuantización son mayores de lo habitual.

El modelo de este ejercicio cabe en una T4 sin cuantización, así que lo entrenamos con LoRA. Primero considera LoRA si cabe en la GPU; si no, QLoRA o un modelo más pequeño. Puedes observar las diferencias de memoria y salida al cargar en 4 bits en la sección 8 (avanzada).

### Ejecutar el entrenamiento

**Qué hacer**: convierte los 14 ejemplos de train al formato de chat (system / user / assistant), añade LoRA y entrena durante 20 épocas (40 pasos). El adaptador entrenado se guarda en `adapter/`.

- `target_modules` indica las capas en las que se inserta LoRA. Se especifican proyecciones de atención (como `q_proj`) y de MLP (como `gate_proj`). De las 24 capas del modelo, 18 son de atención lineal y solo las otras 6 tienen `q_proj` y similares. Todas las capas tienen MLP.
- Lote de 4 × acumulación de gradientes de 2 = 8 ejemplos por paso. Como train tiene 14 ejemplos, cada época tiene 2 pasos (8 ejemplos y 6 ejemplos); 20 épocas suman 40 pasos. Cada ejemplo se usa exactamente 20 veces. El calentamiento y el decaimiento de la tasa de aprendizaje (cosine) se aplican a esos 40 pasos.
- El modelo entrenado queda en la variable `tuned`. Como LoRA se inserta en las capas del modelo fundacional, generar con la variable original `model` también incluye el adaptador. Si necesitas salidas anteriores al entrenamiento, desactiva temporalmente el adaptador como en la sección 6.

**Qué observar**

- `trainable params: ... || all params: ... || trainable%: ...`: una característica de LoRA es entrenar solo un pequeño porcentaje de los parámetros.
- Tabla de `Training Loss` (cada 10 pasos): en el experimento original en japonés empezó alrededor de 1.5 y bajó a 0.01–0.02 en 40 pasos. Estos valores pueden cambiar con los datos traducidos. Si termina sin bajar, comprueba si cambiaste la configuración.
- `Tiempo de entrenamiento: ... segundos | Pasos: 40 | Épocas: 20.0`: en el experimento original en japonés tardó 240–290 segundos en una T4; el tiempo con los datos traducidos puede variar. Los pasos y las épocas confirman que se han ejecutado las 20 pasadas por los 14 ejemplos según el cálculo anterior.

**Advertencias que no impiden continuar**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` indica que la configuración de tokens de terminación se ha adaptado al tokenizador.

**Si algo falla**: si el entrenamiento se interrumpe o no puedes ejecutarlo, lee el apartado 4b siguiente.


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # Pares de entrenamiento: «entrada (system + user)» y «salida deseada (assistant)». Se usa el mismo build_messages que en la evaluación
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# De las 24 capas de Qwen3.5, 18 son de atención lineal; solo las otras 6 tienen q/k/v/o_proj. Todas tienen gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normalmente no se cambia. True usa precisión mixta de 16 bits solo para el cálculo (solo CUDA; pesos en 32 bits). Es más rápido, pero la pérdida puede ser nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Proporción de parámetros que se entrenan con LoRA
t0 = time.time()
trainer.train()  # Véase 4b para reanudar si se interrumpe
print(f"Tiempo de entrenamiento: {time.time()-t0:.0f} segundos | Pasos: {trainer.state.global_step} | Épocas: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # El modo de entrenamiento mantiene dropout activo y hace variar la evaluación (la asignación evita mostrar todo el modelo)

### 4b. Si el entrenamiento se interrumpe o no puedes ejecutarlo

**Se ha interrumpido (por ejemplo, por desconexión del entorno de ejecución)**: los puntos de control se guardan cada 10 pasos en `/content/qlora_lab/ckpt/`. Si los archivos siguen ahí, vuelve a ejecutar las secciones 0–3 y cambia `trainer.train()` de la celda anterior por `trainer.train(resume_from_checkpoint=True)`. Así continuará desde el último punto de control hasta los 40 pasos (20 épocas). Si se ha eliminado el entorno y ya no quedan archivos, empieza de nuevo o usa la alternativa siguiente.

**No puedes entrenar (sin GPU disponible o sin tiempo suficiente)**: continúa la comparación y el análisis con los resultados que obtuvo el docente al ejecutar el mismo cuaderno en una T4 (salidas y puntuaciones posteriores al entrenamiento). Las celdas a partir de la sección 5 no funcionan sin el modelo entrenado `tuned`; lee las explicaciones junto con los resultados del docente. Al entregar, establece `executed_training_myself` de la sección 7 en `False` e indica claramente hasta dónde pudiste ejecutar por tu cuenta.


## 5. Evaluación posterior al entrenamiento (mismo test y misma configuración de generación)

**Qué hacer**: evalúa el modelo entrenado `tuned` con exactamente las mismas 21 preguntas y configuración de generación, y compara antes y después. Lo único que cambia es la presencia del adaptador.

**Qué observar**

- Tabla comparativa: cómo han cambiado `format_rate` (tasa de cumplimiento del formato) y `content_rate` (tasa de acierto del contenido).
- `Regresiones`: preguntas que superaban las pruebas de formato y contenido antes del entrenamiento y fallan después. Lo deseable es que la lista esté vacía; examina las regresiones de tu propio experimento.
- Tabla por preguntas: lee los fallos restantes y distingue su tipo. En el experimento original en japonés se observaron estos casos:
  - Ante una pregunta sin evidencia (la solución de `needs_review` es `True`), el modelo reutilizó otra entrada de la FAQ y afirmó una respuesta.
  - Ante una pregunta con evidencia (la solución es `False`), eludió responder pidiendo confirmación: «No se indica».

En el experimento original en japonés, tras el entrenamiento, las respuestas reproducían casi literalmente el texto de los datos supervisados (el texto original de la FAQ). Se homogeneizaron el formato y la redacción, pero eso no significa necesariamente que el modelo haya aprendido a decidir «si hay evidencia en la FAQ». Comprueba si ocurre lo mismo con los datos traducidos.

Aunque se fije la semilla, el entrenamiento en GPU no se reproduce de forma exacta. Las tasas de cumplimiento del formato y de acierto del contenido pueden ser casi iguales y aun así variar las preguntas que siguen fallando. Es normal que tus «preguntas con fallos restantes» difieran de las de otra persona; compara los tipos de fallos.


In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Preguntas que cumplían formato y contenido antes del entrenamiento y fallan después (regresiones)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Regresiones:", regressed)

# Salidas originales por pregunta. Revisa las comprobaciones de formato (JSON, claves y tipos) y contenido (needs_review y expresiones requeridas)
detail = pd.DataFrame([{"Pregunta": b["question"], "needs_review correcto": b["gold_needs_review"],
                        "Salida antes del entrenamiento": b["raw"], "Antes: formato/contenido": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "Salida después del entrenamiento": a["raw"], "Después: formato/contenido": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. Preguntas de conocimiento (aprender el formato no garantiza el conocimiento)

**Qué hacer**: aparte del experimento de formato, pregunta por 3 hechos que no aparecen en la FAQ ni en los datos de entrenamiento (añadimos como control una pregunta que sí aparece en la FAQ). La respuesta deseada es «Se necesita confirmación (`needs_review: true`)».

Clasificamos las respuestas en 3 grupos.

| Clasificación | Significado |
| --- | --- |
| `format_error` | No respeta el formato JSON (frecuente antes del entrenamiento) |
| `needs_review` | Responde «No se indica. Se necesita confirmación» (respuesta deseada) |
| `asserted` | Afirma una respuesta con `needs_review: false`. Si la pregunta no está en la FAQ, es una afirmación sin evidencia (inventada) |

**Qué observar**: en el experimento original en japonés aparecieron errores de formato antes del entrenamiento y, después, tanto solicitudes de confirmación como afirmaciones. Comprueba qué ocurre con los datos traducidos. La tabla también muestra las salidas originales: si afirma una respuesta, observa en qué parece basarse (si reutiliza otra entrada de la FAQ). Informa de los resultados obtenidos con estos datos, sin concluir de antemano que «el ajuste fino nunca puede corregir el conocimiento».

Las salidas anteriores al entrenamiento se obtienen desactivando temporalmente el adaptador (`tuned.disable_adapter()`).


In [ ]:
KNOWLEDGE_QUESTIONS = [
    "¿Cuándo vence el proyecto final de este curso?",       # No aparece en la FAQ (ficticio)
    "¿Cómo se llama el docente?",                     # No aparece en la FAQ
    "¿Cuál es el tema de la próxima clase?",                 # No aparece en la FAQ
    "¿Cuál es el plazo de entrega de la tarea?",                   # Aparece en la FAQ (control)
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA se inserta en las capas del modelo base; tras entrenar, generar con `model` también incluye el adaptador. Para obtener las salidas anteriores se desactiva temporalmente
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Pregunta": q, "Aparece en la FAQ": q in grounded_questions, "Clasificación antes del entrenamiento": classify(before_raw), "Salida antes del entrenamiento": before_raw,
                 "Clasificación después del entrenamiento": classify(after_raw), "Salida después del entrenamiento": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. Tabla comparativa y entrega

**Qué hacer**: reúne en un único `result.json` la configuración, las versiones de bibliotecas, los resúmenes antes y después del entrenamiento y las preguntas con regresión.

**Qué entregar**

- `result.json`, `eval_before.json` y `eval_after.json` (están en `/content/qlora_lab`; en la lista de archivos de la izquierda, clic derecho → Download. Si `USE_DRIVE = True`, también se copian a Drive)
- La tabla comparativa de la sección 5 y una explicación de los fallos restantes (qué tipos de fallos son)
- Los resultados de la sección 6 (respuestas que pasaron a pedir confirmación y respuestas que siguieron afirmando algo)
- Hasta dónde ejecutaste el entrenamiento por tu cuenta. Si usaste los resultados del docente, establece `executed_training_myself` en `False`.


In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (sin cuantización)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Establece False si usaste los resultados del docente
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Guardado en Drive:", CONFIG["drive_dir"])

## 8. Avanzado: cargar con cuantización en 4 bits y comparar memoria y salidas (opcional)

**Qué hacer**: carga el mismo modelo en 4 bits (NF4) y observa cómo cambian la memoria de los pesos y las salidas anteriores al entrenamiento. QLoRA consiste en añadir LoRA a este modelo base de 4 bits y entrenarlo. Aquí no lo entrenamos.

- bitsandbytes requiere una GPU CUDA, por lo que solo funciona en un entorno de ejecución con GPU. Su instalación tarda aproximadamente 1 minuto.
- La carga en 4 bits usa `device_map="auto"` para colocar el modelo en la GPU (siguiendo el método de carga de bitsandbytes).
- Si pruebas también el entrenamiento en 4 bits, ejecuta primero `prepare_model_for_kbit_training(model_4bit)` y sigue los mismos pasos de la sección 4; evalúa tanto antes como después con el modelo base en 4 bits.

**Qué observar**

- Memoria de los pesos: frente a unos 3GB en 32 bits, en 4 bits ocupa menos de 1GB.
- Tabla comparativa y tabla por preguntas: como no hay entrenamiento, las diferencias con el modelo de 32 bits anterior al entrenamiento (`before`) se deben a la cuantización. En el experimento original en japonés, algunas preguntas mejoraron, mientras que otras fallaron en la comprobación de tipos porque `needs_review` era `"false"` (una cadena), o recibieron afirmaciones como «sí» o «ninguno» aunque no había evidencia. Comprueba si también observas cambios en distintas direcciones con los datos traducidos.


In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Precisión de almacenamiento: 4 bits (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Precisión de cálculo: 16 bits (T4 no admite bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("Memoria de los pesos (GB) | Sin cuantización (32 bits):", BASE_MEMORY_GB, "| 4 bits:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# Sin entrenamiento: la diferencia con el modelo previo sin cuantización (before) se debe a la cuantización
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"Pregunta": r["question"], "needs_review correcto": r["gold_needs_review"], "Salida de 4 bits antes del entrenamiento": r["raw"],
                       "Formato/contenido": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Motivo del fallo de formato": r["format_reason"]} for r in q_rows]))